In [2]:
import os
import numpy as np
import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score
from tqdm import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMAGE_SIZE = 224

# ----------------------------
# TRANSFORMS
# ----------------------------
tf = transforms.Compose([
    transforms.Grayscale(3),
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
])

# ----------------------------
# DATASETS
# ----------------------------
train_ds = datasets.ImageFolder("dataset/train", transform=tf)
val_ds   = datasets.ImageFolder("dataset/val", transform=tf)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=False)
val_loader   = DataLoader(val_ds, batch_size=16, shuffle=False)

# ----------------------------
# FEATURE EXTRACTOR (FROZEN)
# ----------------------------
backbone = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
backbone.fc = nn.Identity()
backbone = backbone.to(DEVICE)
backbone.eval()

@torch.no_grad()
def extract_features(loader):
    feats, labels = [], []
    for x, y in tqdm(loader):
        x = x.to(DEVICE)
        f = backbone(x).cpu().numpy()
        feats.append(f)
        labels.append(y.numpy())
    return np.concatenate(feats), np.concatenate(labels)

# ----------------------------
# EXTRACT FEATURES
# ----------------------------
print("Extracting TRAIN (normal) features...")
X_train, _ = extract_features(train_loader)

print("Extracting VAL features...")
X_val, y_val = extract_features(val_loader)

# In ImageFolder: alphabetical order → defect=0, no_defect=1
# anomaly label: defect=1, normal=0
y_true = (y_val == 0).astype(int)

# ----------------------------
# TRAIN ISOLATION FOREST
# ----------------------------
iso = IsolationForest(
    n_estimators=300,
    contamination=0.1,     # expected defect ratio (adjust if needed)
    random_state=42,
    n_jobs=-1
)

iso.fit(X_train)

# ----------------------------
# ANOMALY SCORES
# ----------------------------
scores = -iso.decision_function(X_val)  # higher = more anomalous

# ----------------------------
# METRIC
# ----------------------------
auroc = roc_auc_score(y_true, scores)
print(f"\n🔥 AUROC = {auroc:.4f}")


Extracting TRAIN (normal) features...


100%|██████████| 697/697 [00:20<00:00, 33.47it/s]


Extracting VAL features...


100%|██████████| 423/423 [00:09<00:00, 42.34it/s]



🔥 AUROC = 0.6118
